# Grok-cv-image-to-3d

Computer vision smoke demo: **image-to-3d** on Kaggle **GPU T4 x2**.

Writes `/kaggle/working/result.json` and prints `SMOKE_OK` on success.


In [ ]:
import json, os, sys, time, traceback, math, gc
from pathlib import Path

import torch
import numpy as np

TASK = os.environ.get("GROK_TASK", "image-to-3d")
NOTEBOOK = "Grok-cv-image-to-3d"
OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

def log(*a):
    print(*a, flush=True)

def gpu_info():
    info = {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
    }
    log("GPU:", info)
    assert info["cuda_available"], "CUDA required — enable Kaggle GPU T4x2"
    assert info["device_count"] >= 2, f"Need T4x2, got device_count={info['device_count']} devices={info.get('devices')}"
    assert all("T4" in d for d in info["devices"]), f"Expected Tesla T4 x2, got {info['devices']}"
    return info

def device0():
    return torch.device("cuda:0")

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_result(payload: dict):
    payload = {
        "ok": True,
        "notebook": NOTEBOOK,
        "task": TASK,
        "domain": "cv",
        **payload,
    }
    path = OUT / "result.json"
    path.write_text(json.dumps(payload, indent=2, default=str))
    log("wrote", path)
    log(json.dumps(payload, indent=2, default=str)[:2000])
    log("SMOKE_OK")
    return payload

def load_sample_image(size=(384, 384)):
    """Download a small sample RGB image (internet on)."""
    from PIL import Image
    import urllib.request
    urls = [
        "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=640",  # cat
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
        "https://picsum.photos/seed/grokcv/512/512",
    ]
    last = None
    for u in urls:
        try:
            fn = OUT / "sample.jpg"
            urllib.request.urlretrieve(u, fn)
            img = Image.open(fn).convert("RGB")
            img = img.resize(size)
            log("sample image", u, img.size)
            return img
        except Exception as e:
            last = e
            log("sample fetch fail", u, e)
    # synthetic fallback
    from PIL import ImageDraw
    img = Image.new("RGB", size, (30, 30, 40))
    d = ImageDraw.Draw(img)
    d.rectangle([40, 40, size[0]-40, size[1]-40], outline=(0, 200, 255), width=6)
    d.ellipse([size[0]//3, size[1]//3, 2*size[0]//3, 2*size[1]//3], fill=(255, 120, 40))
    log("using synthetic sample", last)
    return img

t_start = time.time()
info = gpu_info()



In [ ]:
try:
    # Image-to-3D — depth-based point cloud from monocular depth (T4-safe)
    from transformers import pipeline
    from PIL import Image

    img = load_sample_image((256, 256))
    pipe = pipeline("depth-estimation", model="Intel/dpt-hybrid-midas", device=0)
    t0 = time.time()
    depth = pipe(img)["depth"]
    depth = depth.resize((128, 128))
    d = np.array(depth).astype(np.float32)
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    rgb = np.array(img.resize((128, 128)))
    # back-project
    ys, xs = np.mgrid[0:128, 0:128]
    fx = fy = 128.0
    cx = cy = 64.0
    z = 1.0 + d * 2.0
    x = (xs - cx) * z / fx
    y = (ys - cy) * z / fy
    # subsample
    step = 2
    x, y, z = x[::step, ::step], y[::step, ::step], z[::step, ::step]
    rgb_s = rgb[::step, ::step]
    pts = np.stack([x.reshape(-1), -y.reshape(-1), z.reshape(-1)], axis=1)
    cols = rgb_s.reshape(-1, 3)
    ply = OUT / "img3d.ply"
    with open(ply, "w") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(pts)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n")
        for p, c in zip(pts, cols):
            f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")
    dt = time.time() - t0
    log("wrote", ply, "points", len(pts))
    clear_mem()
    save_result({
        "model": "dpt-hybrid-midas depth back-project",
        "num_points": int(len(pts)),
        "artifact": str(ply),
        "inference_s": dt,
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    })
except Exception as e:
    log("TASK_FAILED", type(e).__name__, e)
    traceback.print_exc()
    err = {
        "ok": False,
        "notebook": NOTEBOOK,
        "task": TASK,
        "error": repr(e),
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    }
    (OUT / "result.json").write_text(json.dumps(err, indent=2, default=str))
    raise
